# Notebook 4: Preprocessing Clarifications for Hyperscanning

In this notebook, we focus on key preprocessing steps for hyperscanning analysis that impact data quality and reliability of inter-brain synchrony measures:

1. **Union vs. Intersection of Epochs and Channels:**  
   Understand the impact of choosing to keep either all valid epochs/channels (union) or only the epochs/channels common to both participants (intersection).

2. **Artifact Rejection:**  
   Use ICA to remove artifacts (e.g., eye blinks, muscle activity) from EEG data and visualize the impact.

3. **Impact on Connectivity Metrics:**  
   Examine how artifact correction affects the resulting connectivity measures.

In [ ]:
import mne
import numpy as np
from scipy.ndimage import gaussian_filter1d
import matplotlib.pyplot as plt
import pickle
from copy import deepcopy
from collections import OrderedDict


# HyPyP modules for preprocessing and analysis
import hypyp.prep as prep  # Contains functions for ICA rejection
import hypyp.analyses as analyses  # For synchrony metrics
import hypyp.viz as viz  # For visualization of results

print("Libraries imported successfully.")

## 1. Union vs. Intersection of Channels

When preprocessing data from two participants, you may encounter situations where different channels are marked as bad for each participant. This presents a critical decision point in the analysis pipeline:

- **Union of Channels:**  
  Keep all channels that are good in at least one participant. This maximizes spatial coverage but may include less reliable channels for one participant.
  
- **Intersection of Channels:**  
  Only keep channels that are good in both participants. This ensures data quality but may reduce spatial coverage.

Below is an example demonstrating both approaches with simulated bad channels.

In [ ]:
# Load preprocessed data from pickle file
with open('./data/hyperscanning_data.pkl', 'rb') as f:
    saved_data = pickle.load(f)
    
# Extract the cleaned epochs from the loaded data
epo1 = saved_data['epo1_clean']
epo2 = saved_data['epo2_clean']

# Print summaries for verification
print("Participant 1 Cleaned Epochs:")
print(epo1)
print("\nParticipant 2 Cleaned Epochs:")
print(epo2)

In [ ]:
# Suppose the raw montage has the following channels (example):
all_channels = epo1.ch_names  # assume both participants share the same channel list

# Mark bad channels:
epo1.info['bads'] = ['Fp1', 'FT9', 'FC5', 'CP2']  # Participant 1 has FP1, FT9, FC5 and CP2 as bad
epo2.info['bads'] = ['F8', 'Cz', 'CP2']   # Participant 2 has F8, Cz and CP2 as bad

# Compute the "good" channels for each participant (union approach)
good_channels_p1 = [ch for ch in all_channels if ch not in epo1.info['bads']]
good_channels_p2 = [ch for ch in all_channels if ch not in epo2.info['bads']]

print("Union Approach:")
print("Participant 1 good channels:", good_channels_p1)
print("Participant 2 good channels:", good_channels_p2)

# Intersection approach: only keep channels that are good in both participants
common_good_channels = list(set(good_channels_p1).intersection(set(good_channels_p2)))
print("\nIntersection Approach:")
print("Common good channels:", sorted(common_good_channels))

# To apply the intersection approach, restrict each epochs object to the common good channels:
epo1_intersect = epo1.copy().pick(common_good_channels)
epo2_intersect = epo2.copy().pick(common_good_channels)

print("\nAfter applying intersection:")
print("Participant 1 channels:", epo1_intersect.ch_names)
print("Participant 2 channels:", epo2_intersect.ch_names)

## 2. Artifact Rejection: ICA for EEG and Union vs. Intersection Strategies

Independent Component Analysis (ICA) is a powerful technique for removing artifacts such as eye blinks and muscle activity from EEG data. When applied to hyperscanning data, we must consider how to handle epochs with artifacts across participants.

After artifact correction (like ICA), additional automatic rejection methods might flag remaining problematic epochs. This introduces another methodological choice: 

- **Union of rejected epochs:** Reject epochs if they are flagged in at least one participant
- **Intersection of rejected epochs:** Reject epochs only if they are flagged in both participants

The visual below illustrates these strategies for epoch rejection.

In [ ]:
def create_didactic_visualization():
    """
    Creates a didactic visualization of the difference between union and intersection strategies.
    """
    # Create simulated data
    n_epochs = 10
    # Rejection masks (True = epoch is rejected)
    p1_rejected = np.zeros(n_epochs, dtype=bool)
    p2_rejected = np.zeros(n_epochs, dtype=bool)
    
    # Participant 1 has epochs 2, 5, 8 rejected
    p1_rejected[2] = p1_rejected[5] = p1_rejected[8] = True
    
    # Participant 2 has epochs 3, 6, 9 rejected
    p2_rejected[3] = p2_rejected[6] = p2_rejected[8] = True
    
    # Calculate union and intersection of REJECTED epochs (not kept)
    # Union of rejected = epochs rejected in at least one participant
    union_rejected = p1_rejected | p2_rejected
    # Intersection of rejected = epochs rejected in both participants
    intersection_rejected = p1_rejected & p2_rejected
    
    # Calculate what epochs remain after applying each strategy
    kept_after_union_rejection = ~union_rejected
    kept_after_intersection_rejection = ~intersection_rejected
    
    # Create the figure
    fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)
    
    # Display the epoch status for each participant (green = kept, red = rejected)
    cmap_rejected = plt.cm.colors.ListedColormap(['green', 'red'])
    
    axes[0].imshow(p1_rejected.reshape(1, -1), cmap=cmap_rejected, aspect='auto')
    axes[0].set_title('Participant 1 (green = kept, red = rejected)')
    axes[0].set_yticks([])
    
    axes[1].imshow(p2_rejected.reshape(1, -1), cmap=cmap_rejected, aspect='auto')
    axes[1].set_title('Participant 2 (green = kept, red = rejected)')
    axes[1].set_yticks([])
    
    # Comparison of strategies
    axes[2].imshow(union_rejected.reshape(1, -1), cmap=cmap_rejected, aspect='auto')
    axes[2].set_title('Union of Rejected (reject if any participant has bad data)')
    axes[2].set_yticks([])
    
    axes[3].imshow(intersection_rejected.reshape(1, -1), cmap=cmap_rejected, aspect='auto')
    axes[3].set_title('Intersection of Rejected (reject only if both participants have bad data)')
    axes[3].set_yticks([])
    
    # Add epoch numbers
    for ax in axes:
        ax.set_xticks(range(n_epochs))
        ax.set_xticklabels(range(1, n_epochs+1))
        ax.grid(True, axis='x', linestyle='--', alpha=0.7)
    
    plt.xlabel('Epoch Number')
    
    # Summary of totals
    plt.figtext(0.5, 0.01, 
                f"Epochs kept - After Union of Rejected: {sum(kept_after_union_rejection)}/{n_epochs}, After Intersection of Rejected: {sum(kept_after_intersection_rejection)}/{n_epochs}",
                ha='center', fontsize=12, 
                bbox=dict(boxstyle="round,pad=0.5", facecolor='white', alpha=0.8))
    
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])  # Adjust for the summary text
    plt.show()

# Call the function to create the didactic visualization
create_didactic_visualization()

## 3. Demonstrating ICA Artifact Removal

To illustrate the effects of ICA on hyperscanning data, we'll introduce artificial artifacts into our cleaned data:

1. **Eye blink artifacts** for Participant 1 (high amplitude, low-frequency signals in frontal channels)
2. **Muscle artifacts** for Participant 2 (high-frequency noise in temporal/central channels)

We'll then apply ICA to remove these artifacts and compare the signals before and after correction.

In [ ]:
# Introduce artifacts to better show differences
def introduce_artifacts(epo1, epo2):
    """
    Artificially introduces artifacts into the EEG data to illustrate
    the difference between union and intersection strategies.
    
    Parameters:
    -----------
    epo1, epo2 : mne.Epochs
        The original clean epoch objects for each participant
        
    Returns:
    --------
    epo1_mod, epo2_mod : mne.Epochs
        Modified epoch objects with artifacts
    artifact_epochs_p1, artifact_epochs_p2 : list
        Lists of epochs where artifacts were introduced
    """
    # Create copies of the Epochs objects to avoid modifying the originals
    epo1_mod = epo1.copy()
    epo2_mod = epo2.copy()
    
    n_epochs = len(epo1)
    n_channels = len(epo1.ch_names)
    
    # Define specific epochs to disturb for each participant
    # For better illustration, we want different rejection patterns
    artifact_epochs_p1 = [0, 2, 4, 6, 8]
    artifact_epochs_p2 = [1, 2, 5, 7, 8]
    
    # For each participant, introduce different types of artifacts
    
    # 1. Participant 1: Add high amplitude artifacts (like eye blinks)
    for epoch_idx in artifact_epochs_p1:
        if epoch_idx < n_epochs:
            # Select frontal channels (typically affected by eye blinks)
            frontal_channels = [i for i, ch in enumerate(epo1.ch_names) 
                               if ch.startswith(('Fp', 'F')) and i < n_channels]
            
            if frontal_channels:
                # Add an artificial "blink"
                for ch_idx in frontal_channels[:3]:  # Limit to a few frontal channels
                    # Create a waveform resembling a blink
                    peak_time = np.random.randint(100, epo1.times.size - 100)
                    artifact = np.zeros(epo1.times.size)
                    artifact[peak_time-50:peak_time+50] = 200 * np.exp(-0.1 * np.abs(np.arange(-50, 50)))
                    
                    # Add the artifact to the data
                    epo1_mod._data[epoch_idx, ch_idx, :] += artifact
    
    # 2. Participant 2: Add high frequency noise (like muscle artifacts)
    for epoch_idx in artifact_epochs_p2:
        if epoch_idx < n_epochs:
            # Select temporal channels (typically affected by muscle artifacts)
            temporal_channels = [i for i, ch in enumerate(epo2.ch_names) 
                                if ch.startswith(('T', 'C')) and i < n_channels]
            
            if temporal_channels:
                # Add artificial "muscle noise"
                for ch_idx in temporal_channels[:3]:  # Limit to a few temporal channels
                    # Create high frequency noise
                    high_freq_noise = np.random.normal(0, 50, epo2.times.size)
                    # Slightly smooth the noise to make it more realistic
                    high_freq_noise = gaussian_filter1d(high_freq_noise, sigma=1)
                    
                    # Apply the noise to only a portion of the epoch
                    start_idx = np.random.randint(0, epo2.times.size // 2)
                    end_idx = start_idx + epo2.times.size // 3  # Duration of the artifact
                    end_idx = min(end_idx, epo2.times.size)  # Make sure not to exceed
                    
                    # Add the artifact to the data
                    epo2_mod._data[epoch_idx, ch_idx, start_idx:end_idx] += high_freq_noise[start_idx:end_idx]
    
    print(f"Artifacts introduced - Participant 1: epochs {artifact_epochs_p1}")
    print(f"Artifacts introduced - Participant 2: epochs {artifact_epochs_p2}")
    
    return epo1_mod, epo2_mod, artifact_epochs_p1, artifact_epochs_p2

In [ ]:
# Clean bad channels for ICA processing
epo1_copy = epo1.copy()
epo2_copy = epo2.copy()
epo1_copy.info['bads'] = []
epo2_copy.info['bads'] = []

# Apply the modification
epo1_mod, epo2_mod, artifact_p1, artifact_p2 = introduce_artifacts(epo1_copy, epo2_copy)

# Apply ICA on the modified data
print("Applying ICA on modified data...")
icas_mod = prep.ICA_fit([
    epo1_mod, epo2_mod
],
    n_components=15,
    method='infomax',
    fit_params=dict(extended=True),
    random_state=42
)

cleaned_epochs_ICA_mod = prep.ICA_choice_comp(icas_mod, [epo1_mod, epo2_mod])
print('ICA correction completed.')

In [ ]:
# Apply AutoReject to further clean the data after ICA correction
print("Applying AutoReject with UNION strategy...")
cleaned_epochs_AR_union, dic_AR_union = prep.AR_local(
    cleaned_epochs_ICA_mod,
    strategy="union",  # Use union strategy for epoch rejection
    threshold=50.0,    # Amplitude threshold for rejection
    verbose=True
)

print(f"Epochs after AutoReject - Participant 1: {len(cleaned_epochs_AR_union[0])}")
print(f"Epochs after AutoReject - Participant 2: {len(cleaned_epochs_AR_union[1])}")

In [ ]:
# Comparing the signal before and after ICA correction
def compare_before_after_ica_and_autoreject(epo1_mod, epo2_mod, cleaned_epochs_ICA_mod):
    """
    Visualize the differences in EEG signals before and after ICA artifact rejection.
    
    Parameters:
    -----------
    epo1_mod, epo2_mod : mne.Epochs
        Epoch objects with artificially introduced artifacts
    cleaned_epochs_ICA_mod : list
        List of epoch objects after ICA correction and AutoReject
    """
    # Get the ICA-cleaned epochs
    epo1_ica = cleaned_epochs_AR_union[0]
    epo2_ica = cleaned_epochs_AR_union[1]
    
    # Set up the figure
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    
    # Choose epochs with artifacts to compare
    artifact_epoch_p1 = artifact_p1[0]  # First artifact epoch for participant 1
    artifact_epoch_p2 = artifact_p2[0]  # First artifact epoch for participant 2
    
    # Select channels with artifacts
    # Frontal channel for participant 1
    frontal_channel_idx = [i for i, ch in enumerate(epo1_mod.ch_names) if ch.startswith(('Fp', 'F'))][0]
    frontal_channel = epo1_mod.ch_names[frontal_channel_idx]
    
    # Temporal channel for participant 2
    temporal_channel_idx = [i for i, ch in enumerate(epo2_mod.ch_names) if ch.startswith(('T', 'C'))][0]
    temporal_channel = epo2_mod.ch_names[temporal_channel_idx]
    
    # Plot data for participant 1
    times = epo1_mod.times
    
    # Before ICA - Participant 1
    data_before = epo1_mod.get_data(copy=True)[artifact_epoch_p1, frontal_channel_idx, :]
    axes[0, 0].plot(times, data_before, 'r', linewidth=2)
    axes[0, 0].set_title(f'Before ICA - Participant 1, Epoch {artifact_epoch_p1+1}, Channel {frontal_channel}')
    axes[0, 0].set_xlabel('Time (s)')
    axes[0, 0].set_ylabel('Amplitude (µV)')
    axes[0, 0].grid(True)
    
    # After ICA - Participant 1
    data_after = epo1_ica.get_data(copy=True)[artifact_epoch_p1, frontal_channel_idx, :]
    axes[0, 1].plot(times, data_after, 'g', linewidth=2)
    axes[0, 1].set_title(f'After ICA - Participant 1, Epoch {artifact_epoch_p1+1}, Channel {frontal_channel}')
    axes[0, 1].set_xlabel('Time (s)')
    axes[0, 1].set_ylabel('Amplitude (µV)')
    axes[0, 1].grid(True)
    
    # Before ICA - Participant 2
    data_before = epo2_mod.get_data(copy=True)[artifact_epoch_p2, temporal_channel_idx, :]
    axes[1, 0].plot(times, data_before, 'r', linewidth=2)
    axes[1, 0].set_title(f'Before ICA - Participant 2, Epoch {artifact_epoch_p2+1}, Channel {temporal_channel}')
    axes[1, 0].set_xlabel('Time (s)')
    axes[1, 0].set_ylabel('Amplitude (µV)')
    axes[1, 0].grid(True)
    
    # After ICA - Participant 2
    data_after = epo2_ica.get_data(copy=True)[artifact_epoch_p2, temporal_channel_idx, :]
    axes[1, 1].plot(times, data_after, 'g', linewidth=2)
    axes[1, 1].set_title(f'After ICA - Participant 2, Epoch {artifact_epoch_p2+1}, Channel {temporal_channel}')
    axes[1, 1].set_xlabel('Time (s)')
    axes[1, 1].set_ylabel('Amplitude (µV)')
    axes[1, 1].grid(True)
    
    plt.tight_layout()
    plt.show()
    
    # Alternative approach: Plot epoch data as a heatmap
    fig, axes = plt.subplots(2, 2, figsize=(18, 10))
    
    # Function to plot a heatmap of epoch data
    def plot_epoch_heatmap(epoch_data, ax, title):
        # Select a single epoch
        data = epoch_data.get_data(copy=True)[0]  # Get first epoch with copy=True
        # Create a heatmap
        im = ax.imshow(data, 
                     aspect='auto', 
                     extent=[epoch_data.times[0], epoch_data.times[-1], 0, data.shape[0]],
                     cmap='RdBu_r',
                     origin='lower')
        ax.set_title(title)
        ax.set_xlabel('Time (s)')
        ax.set_ylabel('Channels')
        ax.set_yticks(np.arange(0.5, data.shape[0], 1))
        ax.set_yticklabels(epoch_data.ch_names)
        return im
    
    # Before ICA - Participant 1
    plot_epoch_heatmap(epo1_mod[artifact_epoch_p1], axes[0, 0], 
                      f'Before ICA - Participant 1, Epoch {artifact_epoch_p1+1}')
    
    # After ICA - Participant 1
    plot_epoch_heatmap(epo1_ica[artifact_epoch_p1], axes[0, 1], 
                      f'After ICA - Participant 1, Epoch {artifact_epoch_p1+1}')
    
    # Before ICA - Participant 2
    plot_epoch_heatmap(epo2_mod[artifact_epoch_p2], axes[1, 0], 
                      f'Before ICA - Participant 2, Epoch {artifact_epoch_p2+1}')
    
    # After ICA - Participant 2
    plot_epoch_heatmap(epo2_ica[artifact_epoch_p2], axes[1, 1], 
                      f'After ICA - Participant 2, Epoch {artifact_epoch_p2+1}')
    
    plt.tight_layout()
    plt.show()

# Call the function to compare signals before and after ICA
compare_before_after_ica_and_autoreject(epo1_mod, epo2_mod, cleaned_epochs_ICA_mod)

## 4. Impact of ICA Correction on Connectivity Metrics

After applying ICA to remove artifacts, it's important to understand how this preprocessing step impacts hyperscanning connectivity measures. Here, we'll compute and compare inter-brain connectivity before and after ICA correction.

We'll use circular correlation in the alpha band as our connectivity metric, which is commonly used in hyperscanning studies to quantify inter-brain synchrony.

In [ ]:
# Compute connectivity metrics for original and ICA-corrected signals
def compute_connectivity_matrices(epo1_mod, epo2_mod, epo1_ica, epo2_ica):
    """
    Computes and compares connectivity matrices before and after ICA correction.
    
    Parameters:
    -----------
    epo1_mod, epo2_mod : mne.Epochs
        Epochs objects before ICA correction
    epo1_ica, epo2_ica : mne.Epochs
        Epochs objects after ICA correction
    """
    # Define frequency bands for analysis
    freq_bands = OrderedDict({
        'Alpha-Low': [7.5, 11],
        'Alpha-High': [11.5, 13]
    })
    
    # Extract sampling rate
    sampling_rate = epo1_mod.info['sfreq']
    
    # Function to compute connectivity for a set of epochs
    def compute_connectivity(epo1, epo2, freq_bands, sampling_rate):
        """
        Computes connectivity metrics between two sets of epochs.
        """
        # Prepare data for connectivity analysis
        dyad_data = np.array([epo1.get_data(copy=True), epo2.get_data(copy=True)])
        
        # Compute analytic signal
        complex_signal = analyses.compute_freq_bands(
            dyad_data, 
            sampling_rate, 
            freq_bands,
            filter_length=int(sampling_rate),
            l_trans_bandwidth=5.0,
            h_trans_bandwidth=5.0
        )
        
        # Compute connectivity using circular correlation
        result = analyses.compute_sync(complex_signal, mode='ccorr', epochs_average=True)
        
        # Get number of channels
        n_ch = len(epo1.info['ch_names'])
        
        # Extract inter-brain connectivity (alpha low band)
        alpha_low = result[0, 0:n_ch, n_ch:2*n_ch]
        
        return alpha_low
    
    # Compute connectivity for original signals
    print("Computing connectivity metrics for ORIGINAL signals...")
    connectivity_original = compute_connectivity(epo1_mod, epo2_mod, freq_bands, sampling_rate)
    
    # Compute connectivity for ICA-corrected signals
    print("Computing connectivity metrics for ICA-CORRECTED signals...")
    connectivity_ica = compute_connectivity(epo1_ica, epo2_ica, freq_bands, sampling_rate)
    
    # Visualize the connectivity matrices
    plt.figure(figsize=(15, 6))
    
    plt.subplot(1, 2, 1)
    im1 = plt.imshow(connectivity_original, cmap='viridis')
    plt.colorbar(im1, label='Connectivity (CCORR)')
    plt.title('Alpha-Low Connectivity - ORIGINAL Signal')
    plt.xlabel('Participant 2 Channels')
    plt.ylabel('Participant 1 Channels')
    
    plt.subplot(1, 2, 2)
    im2 = plt.imshow(connectivity_ica, cmap='viridis')
    plt.colorbar(im2, label='Connectivity (CCORR)')
    plt.title('Alpha-Low Connectivity - ICA-CORRECTED Signal')
    plt.xlabel('Participant 2 Channels')
    plt.ylabel('Participant 1 Channels')
    
    plt.tight_layout()
    plt.show()
    
    # Also calculate the difference between the two to highlight discrepancies
    plt.figure(figsize=(8, 6))
    diff = connectivity_original - connectivity_ica
    im3 = plt.imshow(diff, cmap='coolwarm')
    plt.colorbar(im3, label='Connectivity Difference (Original - ICA)')
    plt.title('Difference in Connectivity (Original - ICA)')
    plt.xlabel('Participant 2 Channels')
    plt.ylabel('Participant 1 Channels')
    plt.tight_layout()
    plt.show()
    
    # Summary statistics of differences
    print("Summary of Connectivity Differences:")
    print(f"Mean absolute difference: {np.mean(np.abs(diff)):.4f}")
    print(f"Maximum absolute difference: {np.max(np.abs(diff)):.4f}")
    print(f"Percentage of channels with differences > 0.05: {np.mean(np.abs(diff) > 0.05) * 100:.1f}%")

# Call the function to compare connectivity matrices
compute_connectivity_matrices(epo1_mod, epo2_mod, cleaned_epochs_AR_union[0], cleaned_epochs_AR_union[1])

## Discussion

Our exploration of preprocessing steps for hyperscanning analysis highlights several important considerations:

- **Union vs. Intersection in Channel Selection:**  
  The choice between union and intersection approaches affects spatial coverage. The intersection approach ensures that all included channels have reliable data from both participants but may reduce spatial coverage. This decision directly impacts the spatial patterns of connectivity that can be detected.

- **Union vs. Intersection in Epoch Rejection:**  
  The choice between union and intersection strategies for epoch rejection has significant implications:
  
  - **Union of rejected epochs** (reject if any participant has bad data) ensures high data quality but typically results in fewer epochs.
  
  - **Intersection of rejected epochs** (reject only if both participants have bad data) retains more epochs but might include data where one participant's signal is compromised.

  These choices are particularly critical in hyperscanning, where we're interested in synchrony between participants. Including epochs where one participant has poor data quality could lead to spurious results.

- **Impact of ICA Correction:**  
  Our comparison demonstrates that artifact correction with ICA can substantially alter the EEG signals and, consequently, the connectivity measures derived from them. Proper artifact removal is essential for reliable hyperscanning results, as artifacts can induce spurious synchrony or mask true neural synchrony.

- **Methodological Choices:**  
  Filtering, baseline correction, and reference selection are often chosen based on previous studies or common practices, but they can greatly influence the analysis outcomes. For reproducibility and transparency, these choices should be systematically documented and their impact on results should be understood.

By carefully considering and documenting these preprocessing steps, researchers can ensure that the results of hyperscanning analysis are both robust and interpretable.

## Conclusion

In this notebook, we have:

1. **Illustrated the difference between union and intersection strategies** in both channel selection and epoch rejection, demonstrating how these choices impact data quality and quantity.

2. **Demonstrated artifact rejection using ICA for EEG data** and visualized its effects on both individual signals and inter-brain connectivity measures.

3. **Compared connectivity metrics before and after artifact correction**, highlighting how preprocessing can substantially alter hyperscanning results.


Key takeaways for hyperscanning preprocessing:

- **Balance between data quality and quantity** is essential; intersection strategies for channels and union strategies for rejected epochs typically favor quality over quantity
- **Artifact removal techniques like ICA** substantially impact connectivity results
- **Document all preprocessing decisions** to ensure reproducibility and allow meaningful comparisons across studies
- **Consider the specific research question** when making preprocessing choices, as different analyses may require different approaches